# 08 — bbq rerun campaign

The shipped test trajectories stall mid-file, so the kill-test's *relaxed
manifold* needed trajectories we ran ourselves. This node is that campaign
(ADR 0005): 418 bbq runs (209 per network), hydrostatic mode at 40 points per
decade from 1e-8 s, on **stock r23.05.1** — deliberately the label
configuration, so the reruns are comparable to the shipped labels rather than
a different physics.

Three families: `shipped` (f1 — same (T, ρ, X₀) as the shipped trajectories,
for comparability), `canonical` (f2 — the kill-test grid), `sobol` (f3 —
training-distribution initial compositions).

Exploratory only — citable values are the RESULTS.md 2026-07-11 rows.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

import nbsupport as nbs
from gnn_nucleo.data.trajectories import load_trajectory, zenodo_trajectory_dir

nbs.style()
QUICK = nbs.QUICK
NETS = ["mesa_80", "mesa_151"]
RERUNS = nbs.REPO / "data" / "bbq_reruns"

In [ ]:
nbs.provenance_header(
    "08",
    "bbq rerun campaign",
    "done",  # no checklist row of its own; ADR 0005 + RESULTS 2026-07-11
    results_rows=[
        "2026-07-11: campaign — 418/418 bbq runs OK (zero failures), ~54 core-h, hydrostatic @ 40 pts/decade from 1e-8 s (ADR 0005; stock r23.05.1 = label config)",
        "2026-07-11: early-time comparability vs shipped at identical (T, ρ, X₀) — species-age agreement median 0.996 / 0.998 (mesa_80/151); shipped-class mid-burn stall artifact ABSENT in reruns",
        "2026-07-11: ingested 171,904 rows/net × {chugunov, unscreened} into FluxStore (hashes in RESULTS.md)",
    ],
    data=[
        "data/bbq_reruns/{mesa_80,mesa_151}/campaign.yaml (authoritative manifest) + per-run output.txt/DONE",
        "data/fluxes/{net}/rerun-trajectories[-unscreened] (FluxStore)",
    ],
    scripts=[
        "scripts/bbq_campaign/make_campaign.py",
        "scripts/bbq_campaign/run_campaign.py",
        "scripts/bbq_campaign/validate_campaign.py",
        "scripts/step6_ingest_reruns.py",
    ],
)

## Load the campaign manifests

`campaign.yaml` is authoritative for run coordinates — run *directory names*
are not parsed (rerun files don't encode (T, ρ) the way shipped ones do).

In [ ]:
camp = {}
for net in NETS:
    d = yaml.safe_load((RERUNS / net / "campaign.yaml").read_text())
    runs = pd.DataFrame(d["runs"])
    runs["done"] = [(RERUNS / net / k / "DONE").exists() for k in runs.run_key]
    camp[net] = runs
    print(f"{net}: {len(runs)} runs, {int(runs.done.sum())} DONE, "
          f"families {dict(Counter(runs.family))}")
    print(f"   config: {d['config']}, {d['points_per_decade']} pts/decade, "
          f"age_start {d['age_start_seconds']:g} s")
print(f"\ntotal across both networks: {sum(len(c) for c in camp.values())} runs "
      f"({int(sum(c.done.sum() for c in camp.values()))} DONE)")
print("Yₑ floor note:", d["ye_floor_note"])

## Figure 1 — campaign coverage against the kill-test grid

The kill-test grid (prediction-target report §6.4) is
T₉ ∈ {1.6, 2.5, 3.3, 4.0, 5.0, 6.3, 7.9} × ρ ∈ {1e7, 1e8, 1e9} × Yₑ ∈ {0.45,
0.48, 0.498}, prioritizing 3.3–5 GK. The `canonical` family targets that grid
directly; `shipped` reproduces the shipped trajectories' coordinates;
`sobol` samples the training distribution.

In [ ]:
FAM_C = {"shipped": "#0072B2", "canonical": "#D55E00", "sobol": "#009E73"}
GRID_T9 = [1.6, 2.5, 3.3, 4.0, 5.0, 6.3, 7.9]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, net in zip(axes, NETS):
    r = camp[net]
    for fam, c in FAM_C.items():
        s = r[r.family == fam]
        ax.scatter(s.t9, s.logRho, c=c, s=34, alpha=0.75, label=f"{fam} (n={len(s)})",
                   edgecolors="none")
    for t in GRID_T9:
        ax.axvline(t, color="0.75", lw=0.7, ls=":", zorder=0)
    for lr in (7, 8, 9):
        ax.axhline(lr, color="0.75", lw=0.7, ls=":", zorder=0)
    ax.axvspan(3.3, 5.0, color="#E69F00", alpha=0.12, zorder=0)
    ax.set_xscale("log")
    ax.set_xlabel("T₉ [GK]")
    ax.set_title(f"{net} — {len(r)} runs, {int(r.done.sum())} DONE")
axes[0].set_ylabel("log₁₀ ρ [g/cm³]")
axes[0].legend(fontsize=8, loc="lower left")
axes[0].text(3.35, 6.75, "kill-test priority\nwindow 3.3–5 GK", fontsize=7.5, color="#8a6d00")
fig.suptitle("bbq rerun campaign coverage (dotted = kill-test grid lines)", y=1.01)
nbs.caption(
    fig,
    "418/418 runs completed with zero failures (~54 core-h total). The canonical family lands on "
    "the kill-test grid nodes; note Yₑ=0.45 is substituted by 0.455 there — the network's own Z/A "
    "floor over A≥12 species (ne22 = 0.4545) makes 0.45 unreachable at fixed composition, a "
    "manifest-recorded design fact, not a miss.",
    results=["RESULTS.md 2026-07-11 campaign rows (418/418 OK, ~54 core-h; ADR 0005)"],
    scripts=["scripts/bbq_campaign/make_campaign.py", "scripts/bbq_campaign/run_campaign.py"],
)

## Figure 2 — completion census and cost

In [ ]:
rt = {}
for net in NETS:
    p = RERUNS / net / "runtimes.csv"
    if p.exists():  # headerless: run_key, wall_seconds, rc, rows, ok
        rt[net] = pd.read_csv(p, names=["run_key", "wall", "rc", "rows", "ok"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), width_ratios=[1, 1.3])
ax = axes[0]
fams = list(FAM_C)
x = np.arange(len(fams))
for i, net in enumerate(NETS):
    counts = [int((camp[net].family == f).sum()) for f in fams]
    done = [int(((camp[net].family == f) & camp[net].done).sum()) for f in fams]
    ax.bar(x + (i - 0.5) * 0.38, counts, 0.36, color=[FAM_C[f] for f in fams],
           alpha=0.45 if i else 0.9, edgecolor="#333333",
           label=f"{net} (all {sum(done)}/{sum(counts)} DONE)")
ax.set_xticks(x, fams)
ax.set_ylabel("runs")
ax.set_title("Runs per family (both nets)")
ax.legend(fontsize=8)

ax = axes[1]
if rt:
    for net, d in rt.items():
        ax.hist(d.wall / 60, bins=30, alpha=0.6, label=f"{net} (Σ {d.wall.sum() / 3600:.1f} core-h)")
    ax.set_xlabel("wall time per run [min]")
    ax.set_ylabel("runs")
    ax.set_title("Per-run cost")
    ax.legend(fontsize=8)
    print("total measured wall:",
          f"{sum(d.wall.sum() for d in rt.values()) / 3600:.1f} core-h "
          f"(RESULTS.md row: ~54 core-h; runtimes.csv accumulates retries)")
nbs.caption(
    fig,
    "Every family completed on both networks. Cost stayed local (~54 core-h) — this is why the "
    "Step-6 CPU allocation ask became Phase-1-only. Contrast the integrator: corpus-scale local Φ "
    "generation is measured INFEASIBLE at ≥1.1e5/4.4e5 core-h (notebook 11).",
    results=["RESULTS.md 2026-07-11 campaign rows (~54 core-h; 418/418 OK)"],
    scripts=["scripts/bbq_campaign/run_campaign.py"],
)

## Figure 3 — shipped vs rerun at identical (T, ρ, X₀)

The comparability check that licenses using reruns in place of shipped rows:
the `shipped` family re-runs the shipped trajectories' own initial
conditions. Early-time evolution must agree (it does: species-age agreement
median 0.996/0.998) — and the mid-burn stall artifact must NOT reappear.

In [ ]:
NET = "mesa_80"
f1 = camp[NET][camp[NET].family == "shipped"].iloc[0]
tr_re = load_trajectory(NET, RERUNS / NET / f1.run_key / "output.txt", source="rerun",
                        logT=f1.logT, logRho=f1.logRho)
tr_sh = load_trajectory(NET, zenodo_trajectory_dir(NET) / f1.comp_source)
print(f"{f1.run_key}: rerun {tr_re.n_rows} rows vs shipped {tr_sh.n_rows} rows "
      f"(logT={f1.logT}, logRho={f1.logRho})")

top = np.argsort(tr_sh.X[0])[::-1][:5]
from gnn_nucleo.graph.isotopes import load_isotope_table  # lazy pynucastro import

names = load_isotope_table(NET).names

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
ax = axes[0]
for i, idx in enumerate(top):
    c = nbs.CYCLE[i % len(nbs.CYCLE)]
    ax.loglog(np.maximum(tr_sh.age, 1e-10), np.maximum(tr_sh.X[:, idx], 1e-12),
              lw=2.2, alpha=0.45, color=c, label=f"{names[idx]} shipped")
    ax.loglog(np.maximum(tr_re.age, 1e-10), np.maximum(tr_re.X[:, idx], 1e-12),
              lw=1.0, ls="--", color=c, label=f"{names[idx]} rerun")
ax.set_xlabel("age [s]")
ax.set_ylabel("mass fraction X")
ax.set_title(f"Top-5 species — {f1.run_key}")
ax.legend(fontsize=6.5, ncol=2)

ax = axes[1]
for tr, label, c in [(tr_sh, "shipped", "#0072B2"), (tr_re, "rerun", "#D55E00")]:
    dX = np.abs(np.diff(tr.X, axis=0)).max(axis=1)
    ax.loglog(np.maximum(tr.age[1:], 1e-10), np.maximum(dX, 1e-18), lw=1, color=c, label=label)
ax.axhline(1e-10, color="#333333", ls="--", lw=1, label="STALL_TOL")
ax.set_xlabel("age [s]")
ax.set_ylabel("max |ΔX| per interval")
ax.set_title("Activity: the shipped mid-burn stall vs the rerun")
ax.legend(fontsize=8)
nbs.caption(
    fig,
    "Quick-look on one f1 pair: the reruns track the shipped trajectory early (measured "
    "species-age agreement median 0.996/0.998 across the family) while resolving the burn on a "
    "denser time grid (40 pts/decade from 1e-8 s). Both eventually reach the same displaced "
    "attractor — the reruns are the LABEL configuration (stock r23.05.1), not corrected physics; "
    "MESA 24.08.1 is the physics witness (notebook 09).",
    results=[
        "RESULTS.md 2026-07-11 early-time comparability rows (median 0.996/0.998; stall artifact absent in reruns)",
    ],
    scripts=["scripts/bbq_campaign/validate_campaign.py"],
)

## TODO (stub)

- **Full per-pair agreement distribution** over all 20 f1 pairs × both nets,
  and the r_QSE single-cluster plateau check (NOT confirmed on clean data:
  0/303, 0/316 rows < 0.1 dex — a negative verdict that retired a Step-5
  deferral). Producer: `scripts/bbq_campaign/validate_campaign.py` (prints;
  persists nothing).

## What this notebook does NOT show

- The fluxes computed from these reruns: notebooks 05/10 (FluxStore runs
  `rerun-trajectories[-unscreened]`, 171,904 rows/net).
- Why both shipped and rerun arrive at a *displaced* attractor: notebook 09.